# Online Retail — 5a, Câu 6: Hot encoding theo hóa đơn

**Võ Thiện Linh — MSSV: 6451071041**

Tạo ma trận số lượng: mỗi dòng là một hóa đơn, mỗi cột là một mặt hàng. Giữ các tổng số lượng (kể cả số lượng âm) và dùng cột thưa để tiết kiệm bộ nhớ.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('../6451071041_LAB1_Bai2_Trang37') / 'Online Retail.xlsx', Path('../Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
germany = df.loc[df['Country'].eq('Germany')]
grouped_quantity = germany.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_index = pd.Index(germany['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
row_position = {invoice: i for i, invoice in enumerate(invoice_index)}
sparse_columns = {}
for description, rows in grouped_quantity.groupby('Description', sort=False):
    values = np.zeros(len(invoice_index), dtype=np.int64)
    positions = rows['InvoiceNo'].map(row_position).to_numpy(dtype=np.intp)
    values[positions] = rows['Quantity'].to_numpy(dtype=np.int64)
    sparse_columns[description] = pd.arrays.SparseArray(values, fill_value=0)
hot_encoding = pd.DataFrame(sparse_columns, index=invoice_index)
print(f'Germany: {hot_encoding.shape[0]:,} hóa đơn × {hot_encoding.shape[1]:,} mặt hàng')
print(f'Bộ nhớ ma trận thưa: {hot_encoding.memory_usage(index=True, deep=True).sum() / 1024**2:.2f} MiB')
print('Một phần ma trận số lượng:')
display(hot_encoding.iloc[:6, :8].sparse.to_dense())